[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/optimization.ipynb)

# MISDA — optimization benchmark

This notebook is a short proof-of-concept for the end-to-end optimization protocol: replace the full objective set by the subset selected by MISDA and ask whether comparable original-space optimization quality is reached in fewer generations.

The comparison is paired:

- **Full** optimizes all original objectives.
- **Reduced** optimizes only the objectives selected by MISDA.
- Both use NSGA-III with the same decision domain, initial decision population, population size, generation budget, and run seed.
- Reduced decision vectors are re-evaluated on the **original M-objective problem** before every comparison.
- This original-space re-evaluation is benchmark instrumentation only; it never feeds back into the Reduced search.

The analytical Pareto front of the original MOP is the common ruler for optimization quality.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test local code. In Colab, install main.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
REMOTE_REF = "main"
target = f"{repo_root}[benchmarks]" if repo_root is not None else f"misda[benchmarks] @ git+https://github.com/monacofj/misda.git@{REMOTE_REF}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import qmc

import misda
import moeabench as mb
from moeabench.core.run import Population

# Short proof-of-concept configuration.
M = 10
SCREEN_POWER = 9          # 2**9 = 512 Sobol points
MISDA_SAMPLE = 2 ** SCREEN_POWER
GT_POINTS = 2000
POPULATION = 60
GENERATIONS = 50
MISDA_SEED = 123
MOEA_SEED = 321
REF_DIRS_SEED = 456
HV_MC_SAMPLES = 20_000


## Experimental protocol

MISDA receives a reproducible **joint Sobol sample** of the original decision domain:

`X_screen → F(X_screen) → MISDA → selected objective subset S`.

All decision variables vary jointly across the screening design. We deliberately do not use one-factor-at-a-time sampling here: this pilot aims to characterize the global objective structure, including interactions, rather than the local response around one fixed operating point. The screening stage is independent of both NSGA-III runs and does not use the analytical Pareto front.

The optimization pair receives one explicitly generated initial decision population `X0`. The same matrix is passed to both NSGA-III instances through pymoo's `sampling=` interface. Reference-direction generation remains separate and dimension-appropriate; only the starting decision population is identical.

For each MOP we then run:

`Full: X → F(X)`

`Reduced: X → F_S(X)`

MISDA changes only the objective set. The decision domain is identical in Full and Reduced. If some decision variables become inactive after objective reduction, that is reported as a consequence of the selected subset; those variables are **not removed from the MOEA**.

The Reduced MOP below is only an objective projection adapter: it delegates all mathematical evaluation to the original MoeaBench MOP and slices the resulting objective matrix. No DTLZ or DPF formula is duplicated.


In [ ]:
class ObjectiveProjectionMOP(mb.mops.BaseMop):
    """Expose a subset of an existing MoeaBench MOP without duplicating it."""

    def __init__(self, source_mop, objective_indices):
        self.source_mop = source_mop
        self.objective_indices = tuple(int(i) for i in objective_indices)
        if len(self.objective_indices) < 2:
            raise ValueError("NSGA-III requires at least two selected objectives.")
        super().__init__(
            name=f"{source_mop.name}[MISDA]",
            M=len(self.objective_indices),
            N=source_mop.N,
            xl=np.asarray(source_mop.xl, dtype=float),
            xu=np.asarray(source_mop.xu, dtype=float),
        )

    def evaluation(self, X, n_ieq_constr=0):
        result = dict(self.source_mop.evaluation(X, n_ieq_constr))
        result["F"] = np.asarray(result["F"], dtype=float)[:, self.objective_indices]
        return result

    def ps(self, n_points=100):
        # Decision-space truth is unchanged; only the exposed objectives differ.
        return self.source_mop.ps(n_points)


In [ ]:
def _nd_front(F):
    """Return the non-dominated subset using MoeaBench's population algebra."""
    return np.asarray(Population(np.asarray(F, dtype=float)).non_dominated().objectives)


def _screen_misda(mop, *, power=SCREEN_POWER, seed=MISDA_SEED):
    """Joint low-discrepancy screening of the complete decision domain."""
    sampler = qmc.Sobol(d=mop.N, scramble=True, seed=seed)
    X_unit = sampler.random_base2(m=power)
    X = qmc.scale(
        X_unit,
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
    )
    F = np.asarray(mop.evaluation(X)["F"], dtype=float)
    frame = pd.DataFrame(F, columns=[f"f{i + 1}" for i in range(mop.M)])
    mis_set = misda.discover(frame, name=f"{mop.name} screening", seed=seed)
    ranking = misda.rank(mis_set)
    selected = ranking.mis()
    return {
        "X": X,
        "F": frame,
        "mis_set": mis_set,
        "ranking": ranking,
        "selected": selected,
        "indices": tuple(int(i) for i in selected.indices),
    }


def _active_decision_dimension(mop, selected_indices):
    """Analytical support count for the two pilot MOPs; never changes the MOEA domain."""
    if mop.__class__.__name__ == "DTLZ2":
        active = set(range(mop.M - 1, mop.N))  # g variables affect every objective
        for i in selected_indices:
            if i == 0:
                active.update(range(mop.M - 1))
            else:
                active.update(range(mop.M - i))
        return len(active)

    if mop.__class__.__name__ == "DPF1":
        # Both base objectives depend on x1 and the shared g term; every projected
        # objective is a linear combination of those base objectives.
        return mop.N

    return None


def _paired_initial_population(mop, *, size=POPULATION, seed=MOEA_SEED):
    """One explicit decision population shared by Full and Reduced."""
    rng = np.random.default_rng(seed)
    return rng.uniform(
        np.asarray(mop.xl, dtype=float),
        np.asarray(mop.xu, dtype=float),
        size=(size, mop.N),
    )


def _canonical_rows(X):
    """Order-independent representation for paired-population assertions."""
    X = np.asarray(X, dtype=float)
    order = np.lexsort(X.T[::-1])
    return X[order]


def _original_space_history(exp, original_mop):
    """Re-evaluate every generation's decision vectors on the original MOP."""
    return [
        _nd_front(original_mop.evaluation(np.asarray(X, dtype=float))["F"])
        for X in exp[0].history("x")
    ]


def _history_evaluations(exp):
    """Population evaluations represented by the recorded generational history."""
    return int(sum(np.asarray(X).shape[0] for X in exp[0].history("x")))


def _metric_value(metric, front, gt):
    return float(metric(np.asarray(front), ref=np.asarray(gt), progress=False))


def _igdplus_history(fronts, gt, label):
    values = [
        _metric_value(mb.metrics.igdplus, front, gt)
        for front in fronts
    ]
    return mb.metrics.MetricMatrix(
        np.asarray(values, dtype=float)[:, None],
        metric_name="IGD+",
        source_name=label,
    )


def _final_metrics(full_front, reduced_front, gt):
    gd_full = _metric_value(mb.metrics.gdplus, full_front, gt)
    gd_reduced = _metric_value(mb.metrics.gdplus, reduced_front, gt)
    igd_full = _metric_value(mb.metrics.igdplus, full_front, gt)
    igd_reduced = _metric_value(mb.metrics.igdplus, reduced_front, gt)

    hv_kwargs = dict(
        ref=np.asarray(gt),
        mode="auto",
        scale="abs",
        n_samples=HV_MC_SAMPLES,
        mc_seed=MOEA_SEED,
        progress=False,
    )
    hv_full = float(mb.metrics.hypervolume(np.asarray(full_front), **hv_kwargs))
    hv_reduced = float(mb.metrics.hypervolume(np.asarray(reduced_front), **hv_kwargs))

    return pd.DataFrame(
        {
            "Full": [gd_full, igd_full, hv_full],
            "Reduced": [gd_reduced, igd_reduced, hv_reduced],
            "Reduced - Full": [
                gd_reduced - gd_full,
                igd_reduced - igd_full,
                hv_reduced - hv_full,
            ],
        },
        index=["GD+", "IGD+", "HV"],
    )


In [ ]:
optimization_results = {}


def run_optimization_case(name, mop):
    screening = _screen_misda(mop)
    selected_indices = screening["indices"]
    selected_labels = [f"f{i + 1}" for i in selected_indices]
    active_decision_dimension = _active_decision_dimension(mop, selected_indices)

    reduced_mop = ObjectiveProjectionMOP(mop, selected_indices)

    # Pairing is explicit: both formulations receive exactly the same X0.
    X0 = _paired_initial_population(mop)
    initial_original = np.asarray(mop.evaluation(X0)["F"], dtype=float)

    full = mb.experiment(
        mop=mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            ref_dirs_seed=REF_DIRS_SEED,
            sampling=X0.copy(),
        ),
    )
    full.name = f"{name} — Full"

    reduced = mb.experiment(
        mop=reduced_mop,
        moea=mb.moeas.NSGA3(
            population=POPULATION,
            generations=GENERATIONS,
            seed=MOEA_SEED,
            ref_dirs_seed=REF_DIRS_SEED,
            sampling=X0.copy(),
        ),
    )
    reduced.name = f"{name} — Reduced"

    full.run(repeat=1, silent=True)
    reduced.run(repeat=1, silent=True)

    # NSGA-III may reorder the initial population during its objective-specific
    # survival step. Membership, not row order, is the pairing invariant.
    np.testing.assert_allclose(
        _canonical_rows(full[0].history("x")[0]),
        _canonical_rows(X0),
    )
    np.testing.assert_allclose(
        _canonical_rows(reduced[0].history("x")[0]),
        _canonical_rows(X0),
    )

    # Pareto ground truth belongs to the original M-objective problem and is
    # used only as the common ruler for optimization quality.
    gt_exp = mb.experiment(mop=mop)
    gt = np.asarray(gt_exp.optimal(n_points=GT_POINTS).objectives)

    full_history = _original_space_history(full, mop)
    reduced_history = _original_space_history(reduced, mop)
    full_front = full_history[-1]
    reduced_front = reduced_history[-1]

    metrics = _final_metrics(full_front, reduced_front, gt)

    diag_full = mb.clinic.audit(
        full_front,
        ground_truth=gt,
        initial_data=initial_original,
        problem=name,
        k=POPULATION,
    )
    diag_full.experiment_name = "Full"

    diag_reduced = mb.clinic.audit(
        reduced_front,
        ground_truth=gt,
        initial_data=initial_original,
        problem=name,
        k=POPULATION,
    )
    diag_reduced.experiment_name = "Reduced"

    igd_full = _igdplus_history(full_history, gt, "Full")
    igd_reduced = _igdplus_history(reduced_history, gt, "Reduced")

    full_evaluations = _history_evaluations(full)
    reduced_evaluations = _history_evaluations(reduced)
    assert full_evaluations == reduced_evaluations

    result = {
        "name": name,
        "mop": mop,
        "screening": screening,
        "selected_indices": selected_indices,
        "selected_labels": selected_labels,
        "active_decision_dimension": active_decision_dimension,
        "initial_X": X0,
        "initial_original": initial_original,
        "reduced_mop": reduced_mop,
        "full": full,
        "reduced": reduced,
        "gt": gt,
        "full_history": full_history,
        "reduced_history": reduced_history,
        "full_front": full_front,
        "reduced_front": reduced_front,
        "metrics": metrics,
        "diag_full": diag_full,
        "diag_reduced": diag_reduced,
        "igd_full": igd_full,
        "igd_reduced": igd_reduced,
        "evaluations": full_evaluations,
    }
    optimization_results[name] = result

    print(f"{name}: MISDA reduction {mop.M} → {len(selected_indices)}")
    print(f"Selected objectives: {', '.join(selected_labels)}")
    if active_decision_dimension is not None:
        print(
            f"Decision dimension: original={mop.N}, "
            f"active after objective reduction={active_decision_dimension} "
            "(measured only; MOEA domain unchanged)"
        )
    print(
        f"Budget per treatment: generations={len(full[0].history('x'))}, "
        f"evaluations={full_evaluations}"
    )
    display(metrics)

    mb.view.topology(
        full_front,
        reduced_front,
        gt=gt,
        show_gt=True,
        objectives=[0, 1, 2],
        labels=["Full", "Reduced"],
        title=f"{name}: Full vs Reduced in original objective space (f1–f3 projection)",
    )

    mb.view.radar(
        diag_full,
        diag_reduced,
        title=f"{name}: clinical quality in original objective space",
    )

    mb.view.history(
        igd_full,
        igd_reduced,
        title=f"{name}: IGD+ convergence in original objective space",
    )

    return result


## Pilot battery

This first run is intentionally short. It is a proof of concept for the complete protocol, not a definitive performance study.

- **DTLZ2** — regular smooth, non-degenerate control. MISDA's existing classical benchmark deliberately does **not** assign DTLZ2 a MISDA-specific latent/structural ground truth; its known Pareto manifold dimension is external geometric context.
- **DPF1** — explicit degenerate projection from an intrinsic base of `D=2` objectives to `M=10`. This construction dimension is useful external context, but it is not silently treated as a declared MISDA structural truth.

A separate notion of ground truth is used for optimization quality: MoeaBench's analytical Pareto set/front of the original MOP. That Pareto ground truth is the common reference for GD+, IGD+, HV, and the clinical Q-scores.

If the pilot behaves sensibly, the remaining planned cases (DTLZ5, DTLZ7, DPF3, DPF5) can be added afterwards.


In [ ]:
PROBLEMS = {
    "DTLZ2": mb.mops.DTLZ2(M=M),
    "DPF1": mb.mops.DPF1(M=M, D=2, K=5),
}

PROBLEM_CONTEXT = {
    "DTLZ2": {
        "misda_truth": None,
        "external_context": f"Pareto manifold dimension = {M - 1}",
    },
    "DPF1": {
        "misda_truth": None,
        "external_context": "Intrinsic base objective dimension D = 2",
    },
}

for name, mop in PROBLEMS.items():
    print(f"{name}: M={mop.M}, N={mop.N} — {PROBLEM_CONTEXT[name]['external_context']}")


## DTLZ2 — regular smooth control

The non-degenerate control checks that MISDA does not create an artificial optimization advantage when little or no objective reduction is justified.


In [ ]:
dtlz2 = run_optimization_case("DTLZ2", PROBLEMS["DTLZ2"])


## DPF1 — linear degenerate projection

DPF1 is a positive control for high-dimensional objectives generated from a low-dimensional base front.


In [ ]:
dpf1 = run_optimization_case("DPF1", PROBLEMS["DPF1"])


# Suite summary

The summary keeps the scientific axes separate: reduction obtained, final original-space quality, and convergence. A lower GD+/IGD+ and a higher HV indicate better final approximation, but the table deliberately reports values and deltas rather than declaring a winner.


In [ ]:
summary_rows = []
for name, result in optimization_results.items():
    metrics = result["metrics"]
    summary_rows.append(
        {
            "Problem": name,
            "Objectives": result["mop"].M,
            "Selected": len(result["selected_indices"]),
            "Decision dim.": result["mop"].N,
            "Active decision dim.": result["active_decision_dimension"],
            "Generations": len(result["full"][0].history("x")),
            "Evaluations": result["evaluations"],
            "GD+ Full": metrics.loc["GD+", "Full"],
            "GD+ Reduced": metrics.loc["GD+", "Reduced"],
            "IGD+ Full": metrics.loc["IGD+", "Full"],
            "IGD+ Reduced": metrics.loc["IGD+", "Reduced"],
            "HV Full": metrics.loc["HV", "Full"],
            "HV Reduced": metrics.loc["HV", "Reduced"],
            "Final IGD+ Δ": metrics.loc["IGD+", "Reduced - Full"],
        }
    )

optimization_summary = pd.DataFrame(summary_rows)
optimization_summary
